# ALTCBoost: temporal fraud benchmark

Attach the Kaggle dataset `mlg-ulb/creditcardfraud` using Add Input, then Run All on CPU.
This notebook contains the model source, tests, discovery and benchmark runner.
It creates aggregate outputs only. It does not export transaction rows or predictions.

The proposed aligned variant removes hard-negative weighting and selects at 0.1% FPR.
It is an ablation whose performance must be measured. It is not an exact partial-AUC optimizer.

The dataset covers a short historical window and provides no card identities or label arrival times.
Time-ordered evaluation cannot establish entity independence, label-delay realism or production performance.


In [ ]:
from pathlib import Path
import os, sys, json
root=Path("/kaggle/working/altcboost_experiment")
root.mkdir(parents=True,exist_ok=True)
files={'altcboost.py': '"""ALTCBoost — experimental Alternating Linear, Tree and Cross boosting classifier.\n\nResearch prototype for numeric tabular binary classification. Not claimed to be\nnovel relative to all published literature, or universally better than XGBoost.\n\nAlgorithm:\n  1. Fit a regularized logistic (global linear) prior.\n  2. Correct logits using 2nd-order (Newton/IRLS) regression trees.\n  3. After every tree, greedily choose one explicit pairwise interaction using\n     second-order gain, update the logit margin with an L2-regularized step.\n  4. Use a held-out validation set for early stopping.\n\nNo XGBoost used in the proposed model itself.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Optional\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.tree import DecisionTreeRegressor\nfrom sklearn.model_selection import train_test_split\nfrom scipy.special import expit, logit\nfrom sklearn.metrics import log_loss\n\n\nclass ALTCBoostClassifier:\n    def __init__(self, n_estimators=90, learning_rate=0.08, max_leaf_nodes=9,\n                 min_samples_leaf=35, linear_C=1.0, pair_penalty=10.0,\n                 pair_rate=0.35, early_stopping_rounds=16, random_state=42,\n                 use_cross=True):\n        self.n_estimators=n_estimators\n        self.learning_rate=learning_rate\n        self.max_leaf_nodes=max_leaf_nodes\n        self.min_samples_leaf=min_samples_leaf\n        self.linear_C=linear_C\n        self.pair_penalty=pair_penalty\n        self.pair_rate=pair_rate\n        self.early_stopping_rounds=early_stopping_rounds\n        self.random_state=random_state\n        self.use_cross=use_cross\n\n    def _features(self, X):\n        X=np.asarray(X,dtype=float)\n        if X.ndim != 2: raise ValueError(\'X must be a 2D numeric array\')\n        return np.clip(self.scaler_.transform(X),-6,6)\n\n    def _cross_matrix(self,X):\n        i,j=self.pairs_[:,0],self.pairs_[:,1]\n        C=X[:,i]*X[:,j]\n        # subtract train-time means and divide train-time standard deviations\n        return np.clip((C-self.cross_means_)/self.cross_stds_,-10,10)\n\n    @staticmethod\n    def _loss(y, logits):\n        return float(np.mean(np.logaddexp(0, logits)-y*logits))\n\n    def fit(self, X, y, X_val=None, y_val=None):\n        X=np.asarray(X,dtype=float)\n        y=np.asarray(y,dtype=int)\n        if not np.all(np.isfinite(X)): raise ValueError(\'Only finite numerical X supported in prototype.\')\n        if X_val is None:\n            X,X_val,y,y_val=train_test_split(X,y,test_size=0.22,stratify=y,random_state=self.random_state)\n        else:\n            X_val=np.asarray(X_val,dtype=float)\n            y_val=np.asarray(y_val,dtype=int)\n        if np.unique(y).size!=2: raise ValueError(\'binary y with both classes required\')\n        self.scaler_=StandardScaler().fit(X)\n        Xt=self._features(X); Xv=self._features(X_val)\n        self.linear_=LogisticRegression(C=self.linear_C, solver=\'lbfgs\',max_iter=500,random_state=self.random_state).fit(Xt,y)\n        F=self.linear_.decision_function(Xt).astype(float)\n        Fv=self.linear_.decision_function(Xv).astype(float)\n        self.steps_=[]\n        self.best_iteration_=0\n        best_loss=self._loss(y_val,Fv)\n        best_idx=0\n        patience=0\n        self.pairs_=np.array([(i,j) for i in range(X.shape[1]) for j in range(i+1,X.shape[1])],dtype=int).reshape(-1,2)\n        if self.pairs_.size and self.use_cross:\n            i,j=self.pairs_[:,0],self.pairs_[:,1]\n            Ct0=Xt[:,i]*Xt[:,j]\n            self.cross_means_=Ct0.mean(axis=0)\n            self.cross_stds_=np.maximum(Ct0.std(axis=0),.05)\n            Ct=self._cross_matrix(Xt); Cv=self._cross_matrix(Xv)\n        else:\n            self.cross_means_=np.array([])\n            self.cross_stds_=np.array([])\n            Ct=Cv=None\n        for t in range(self.n_estimators):\n            p=expit(np.clip(F,-25,25))\n            g=y-p\n            h=np.maximum(p*(1-p),0.002)\n            newton=np.clip(g/h,-15,15)\n            tree=DecisionTreeRegressor(max_leaf_nodes=self.max_leaf_nodes,\n                 min_samples_leaf=self.min_samples_leaf,random_state=self.random_state+t)\n            tree.fit(Xt,newton,sample_weight=h)\n            dtr=np.clip(tree.predict(Xt),-5,5)*self.learning_rate\n            dval=np.clip(tree.predict(Xv),-5,5)*self.learning_rate\n            F+=dtr; Fv+=dval\n            cross_info=None\n            if Ct is not None and Ct.shape[1]:\n                p=expit(np.clip(F,-25,25))\n                g=y-p; h=np.maximum(p*(1-p),0.002)\n                # one-coordinate Newton gain, optimized on the current residual\n                numerator=Ct.T@g\n                denominator=(Ct**2).T@h+self.pair_penalty\n                scores=numerator*numerator/denominator\n                j=int(np.argmax(scores))\n                coeff=np.clip(numerator[j]/denominator[j],-3,3)*self.learning_rate*self.pair_rate\n                F+=coeff*Ct[:,j]\n                Fv+=coeff*Cv[:,j]\n                cross_info=(j, float(coeff))\n            self.steps_.append((tree,cross_info))\n            val_loss=self._loss(y_val,Fv)\n            if val_loss<best_loss-1e-5:\n                best_loss=val_loss\n                best_idx=t+1\n                patience=0\n            else:\n                patience+=1\n            if patience >= self.early_stopping_rounds:\n                break\n        self.best_iteration_=best_idx\n        self.validation_logloss_=best_loss\n        return self\n\n    def decision_function(self, X):\n        Xt=self._features(X)\n        F=self.linear_.decision_function(Xt).astype(float)\n        Ct=self._cross_matrix(Xt) if self.use_cross and self.pairs_.size else None\n        for tree, cross_info in self.steps_[:self.best_iteration_]:\n            F+=self.learning_rate*np.clip(tree.predict(Xt),-5,5)\n            if cross_info is not None:\n                j,coeff=cross_info\n                F+=coeff*Ct[:,j]\n        return F\n\n    def predict_proba(self,X):\n        p=expit(np.clip(self.decision_function(X),-35,35))\n        return np.column_stack((1-p,p))\n\n    def predict(self,X):\n        return (self.predict_proba(X)[:,1]>.5).astype(int)\n', 'altcboost_v2.py': '"""ALTCBoost 2.0 research prototype: exact compiled inference + FPR operating point.\n\nFit uses the original ALTCBoost algorithm (same training and ranking). Compiles\ntrained coefficients and tree structure into Numba CPU kernels for single-event\ninference; builds optional safe branch-and-bound classification using tree leaf\nbounds. Calibrates an empirical operating threshold from *separate* labeled\ncalibration data. No claim of population FPR guarantee or universal superiority.\n"""\nfrom __future__ import annotations\nimport numpy as np\nfrom scipy.special import expit, logit\nfrom scipy.stats import beta\nfrom numba import njit\nfrom altcboost import ALTCBoostClassifier\n\n\n@njit(cache=True)\ndef _score(X, means, scales, coef, intercept, ci, cj, cm, cs, cc,\n           offsets, left, right, feat, threshold, vals, lr):\n    out = np.empty(X.shape[0], dtype=np.float64)\n    for row in range(X.shape[0]):\n        z = np.empty(X.shape[1], dtype=np.float64)\n        m = intercept\n        for j in range(X.shape[1]):\n            v = (X[row,j] - means[j]) / scales[j]\n            z[j] = min(6.0, max(-6.0, v))\n            m += coef[j] * z[j]\n        for j in range(len(ci)):\n            c = (z[ci[j]] * z[cj[j]] - cm[j]) / cs[j]\n            m += cc[j] * min(10.0, max(-10.0, c))\n        for t in range(len(offsets)-1):\n            base = offsets[t]\n            node = base\n            while left[node] != -1:\n                # sklearn converts tree inputs to float32 before comparing splits.\n                if np.float32(z[feat[node]]) <= threshold[node]:\n                    node = base + left[node]\n                else:\n                    node = base + right[node]\n            m += lr * min(5.0,max(-5.0,vals[node]))\n        out[row] = m\n    return out\n\n\n@njit(cache=True)\ndef _classify_early(X, decision_logit, means, scales, coef, intercept,\n                    ci, cj, cm, cs, cc, offsets, left, right, feat,\n                    threshold, vals, lr, suffix_min, suffix_max):\n    out = np.zeros(X.shape[0], dtype=np.uint8)\n    trees_evaluated = np.zeros(X.shape[0],dtype=np.int32)\n    for row in range(X.shape[0]):\n        z = np.empty(X.shape[1], dtype=np.float64)\n        m = intercept\n        for j in range(X.shape[1]):\n            v = (X[row,j] - means[j]) / scales[j]\n            z[j] = min(6.0,max(-6.0,v))\n            m += coef[j]*z[j]\n        for j in range(len(ci)):\n            c = (z[ci[j]]*z[cj[j]]-cm[j])/cs[j]\n            m += cc[j]*min(10.0,max(-10.0,c))\n        for t in range(len(offsets)-1):\n            if m+suffix_max[t] < decision_logit:\n                break\n            if m+suffix_min[t] >= decision_logit:\n                out[row]=1\n                break\n            base=offsets[t]\n            node=base\n            while left[node]!=-1:\n                if np.float32(z[feat[node]]) <= threshold[node]:\n                    node=base+left[node]\n                else:\n                    node=base+right[node]\n            m += lr*min(5.0,max(-5.0,vals[node]))\n            trees_evaluated[row]+=1\n        else:\n            out[row]=int(m>=decision_logit)\n    return out,trees_evaluated\n\n\nclass ALTCBoostV2(ALTCBoostClassifier):\n    """Numerically compatible accelerated prediction of an ALTCBoost model."""\n    def fit(self, X, y, X_val=None, y_val=None):\n        super().fit(X,y,X_val,y_val)\n        for name in list(vars(self)):\n            if name == \'decision_threshold_\' or name.startswith(\'calibration_\'):\n                delattr(self, name)\n        return self.compile()\n\n    def compile(self):\n        self._means=np.asarray(self.scaler_.mean_,dtype=np.float64)\n        self._scales=np.asarray(self.scaler_.scale_,dtype=np.float64)\n        self._coef=np.asarray(self.linear_.coef_[0],dtype=np.float64)\n        self._intercept=float(self.linear_.intercept_[0])\n        # Collapse repeated pairwise updates into a single coefficient / unique pair.\n        agg={}\n        for tree,cross in self.steps_[:self.best_iteration_]:\n            if cross is not None:\n                idx, delta=cross\n                agg[idx]=agg.get(idx,0.)+delta\n        ids=np.array(list(agg),dtype=np.int32)\n        self._ci=np.asarray(self.pairs_[ids,0] if len(ids) else [],dtype=np.int32)\n        self._cj=np.asarray(self.pairs_[ids,1] if len(ids) else [],dtype=np.int32)\n        self._cm=np.asarray(self.cross_means_[ids] if len(ids) else [],dtype=np.float64)\n        self._cs=np.asarray(self.cross_stds_[ids] if len(ids) else [],dtype=np.float64)\n        self._cc=np.asarray(list(agg.values()),dtype=np.float64)\n        offsets=[0]; left=[];right=[]; feat=[];thr=[];vals=[]\n        mins=[];maxs=[]\n        for tree,_ in self.steps_[:self.best_iteration_]:\n            nodes=tree.tree_\n            l=nodes.children_left.astype(np.int32)\n            r=nodes.children_right.astype(np.int32)\n            v=nodes.value[:,0,0].astype(np.float64)\n            left.extend(l);right.extend(r)\n            feat.extend(nodes.feature.astype(np.int32))\n            thr.extend(nodes.threshold.astype(np.float64));vals.extend(v)\n            offsets.append(offsets[-1]+len(v))\n            leafvals=np.clip(v[l==-1],-5.,5.)*self.learning_rate\n            mins.append(leafvals.min());maxs.append(leafvals.max())\n        self._offsets=np.asarray(offsets,dtype=np.int32)\n        self._left=np.asarray(left,dtype=np.int32)\n        self._right=np.asarray(right,dtype=np.int32)\n        self._feat=np.asarray(feat,dtype=np.int32)\n        self._thr=np.asarray(thr,dtype=np.float64)\n        self._vals=np.asarray(vals,dtype=np.float64)\n        self._suffix_min=np.r_[np.cumsum(mins[::-1])[::-1],0.].astype(np.float64)\n        self._suffix_max=np.r_[np.cumsum(maxs[::-1])[::-1],0.].astype(np.float64)\n        return self\n\n    def _args(self):\n        return (self._means,self._scales,self._coef,self._intercept,\n                self._ci,self._cj,self._cm,self._cs,self._cc,self._offsets,\n                self._left,self._right,self._feat,self._thr,self._vals,float(self.learning_rate))\n\n    def decision_function(self,X):\n        X=np.asarray(X,dtype=np.float64)\n        if X.ndim != 2 or X.shape[1] != len(self._means) or not np.isfinite(X).all():\n            raise ValueError(\'Input must be finite with the fitted feature count\')\n        return _score(X,*self._args())\n\n    def predict_proba(self,X):\n        p=expit(np.clip(self.decision_function(X),-35.,35.))\n        return np.column_stack((1.-p,p))\n\n    def fit_threshold(self, X_cal, y_cal, max_fpr=0.01, confidence=None):\n        """Conservative empirical FPR cap, not a future false-positive guarantee.\n\n        Use a calibration set distinct from train, early-stop selection, and test.\n        """\n        if not 0.<=max_fpr<1.:\n            raise ValueError(\'max_fpr must be between 0 and 1\')\n        labels=np.asarray(y_cal,dtype=np.int32)\n        neg=np.sort(self.predict_proba(X_cal)[labels==0,1])[::-1]\n        if len(neg)==0: raise ValueError(\'No negative calibration examples\')\n        allow=int(np.floor(max_fpr*len(neg)))\n        if confidence is not None:\n            if not 0. < confidence < 1.:\n                raise ValueError(\'confidence must be in (0,1)\')\n            # One-sided order-statistic tolerance bound using Beta tails.\n            # Requires calibration negatives representative of future cases.\n            permitted=[k for k in range(allow+1)\n                       if float(beta.ppf(confidence,k+1,len(neg)-k)) <= max_fpr]\n            if not permitted:\n                raise ValueError(\'Not enough negative calibration samples for a\'\n                                 \' statistically supported FPR bound at this confidence\')\n            allow=max(permitted)\n        if allow>=len(neg):\n            raise ValueError(\'max_fpr too large\')\n        self.decision_threshold_=float(np.nextafter(neg[allow],np.inf))\n        self.calibration_n_negative_=len(neg)\n        self.calibration_max_fpr_=max_fpr\n        self.calibration_confidence_=confidence\n        self.calibration_allowed_false_positives_=allow\n        return self\n\n    def predict_flags(self,X,threshold=None):\n        if threshold is None:\n            threshold=getattr(self,\'decision_threshold_\',None)\n        if threshold is None:\n            raise ValueError(\'fit_threshold or pass a threshold before classification\')\n        if not 0.<threshold<1.: raise ValueError(\'threshold should be within (0,1)\')\n        X=np.asarray(X,dtype=np.float64)\n        if X.ndim != 2 or X.shape[1] != len(self._means) or not np.isfinite(X).all():\n            raise ValueError(\'Input must be finite with the fitted feature count\')\n        return _classify_early(X,float(logit(threshold)),*self._args(),\n                               self._suffix_min,self._suffix_max)\n', 'altcboost_v3.py': '"""ALTCBoost V3: a *research prototype* of hard-negative-aware linear/tree/cross boosting.\n\nResearch ingredients motivated by partial ROC-AUC and Neyman-Pearson literature:\n    - Explicit emphasis on the current high-scoring negative examples in boosted\n      Newton updates (adaptive weights; *not* exact partial AUC minimization).\n    - Validation model selection trades average precision and standardized\n      partial ROC-AUC at a small false-positive rate.\n    - Independent calibration via the inherited V2 method.\n    - Compact compiled Numba inference and exact decision early exit from V2.\n\nThis is an experimental combination of known ideas, NOT a claim of a novel\npublished algorithm or superiority to strong established baselines.\n"""\nfrom __future__ import annotations\nimport numpy as np\nfrom scipy.special import expit\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.tree import DecisionTreeRegressor\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom altcboost_v2 import ALTCBoostV2\n\n\nclass ALTCBoostV3(ALTCBoostV2):\n    def __init__(self, *, hard_negative_strength=2.5,\n                 hard_negative_fraction=0.08,\n                 hard_negative_temperature=0.35,\n                 hard_negative_warmup=12,\n                 low_fpr_validation=0.05,\n                 validation_ap_weight=0.5,\n                 n_estimators=90, learning_rate=0.08, max_leaf_nodes=9,\n                 min_samples_leaf=35, linear_C=1.0, pair_penalty=10.0,\n                 pair_rate=0.35, early_stopping_rounds=16, random_state=42,\n                 use_cross=True):\n        super().__init__(n_estimators=n_estimators,\n            learning_rate=learning_rate,max_leaf_nodes=max_leaf_nodes,\n            min_samples_leaf=min_samples_leaf,linear_C=linear_C,\n            pair_penalty=pair_penalty,pair_rate=pair_rate,\n            early_stopping_rounds=early_stopping_rounds,\n            random_state=random_state,use_cross=use_cross)\n        if not (0 < hard_negative_fraction <= 0.5):\n            raise ValueError(\'hard_negative_fraction must be in (0, .5]\')\n        if hard_negative_strength < 0 or hard_negative_temperature <= 0:\n            raise ValueError(\'invalid hard negative weighting\')\n        if not 0 < low_fpr_validation <= 1:\n            raise ValueError(\'low_fpr_validation must be in (0,1]\')\n        if not 0 <= validation_ap_weight <= 1:\n            raise ValueError(\'validation_ap_weight must be in [0,1]\')\n        self.hard_negative_strength=float(hard_negative_strength)\n        self.hard_negative_fraction=float(hard_negative_fraction)\n        self.hard_negative_temperature=float(hard_negative_temperature)\n        self.hard_negative_warmup=int(hard_negative_warmup)\n        self.low_fpr_validation=float(low_fpr_validation)\n        self.validation_ap_weight=float(validation_ap_weight)\n\n    def _validation_objective(self, y, logits):\n        # standardized partial ROC-AUC by sklearn; the max_fpr is ONLY used\n        # for model selection, NOT a guarantee of future operating-point FPR.\n        ap=average_precision_score(y, logits)\n        if self.low_fpr_validation < 1:\n            pauc=roc_auc_score(y, logits, max_fpr=self.low_fpr_validation)\n        else:\n            pauc=roc_auc_score(y,logits)\n        return self.validation_ap_weight*ap+(1-self.validation_ap_weight)*pauc\n\n    def _weights(self,y,F,iteration):\n        w=np.ones_like(F)\n        if self.hard_negative_strength<=0 or iteration<self.hard_negative_warmup:\n            return w\n        negatives=(y==0)\n        # Train-only order statistic. Never uses validation, calibration, or test.\n        cutoff=np.quantile(F[negatives],1-self.hard_negative_fraction)\n        softness=expit(np.clip((F[negatives]-cutoff)/self.hard_negative_temperature,-30,30))\n        w[negatives]+=self.hard_negative_strength*softness\n        # Average weight normalization does not change Newton ratios, but avoids\n        # exploding regularization interactions during training.\n        return w / w.mean()\n\n    def fit(self,X,y,X_val=None,y_val=None):\n        X=np.asarray(X,dtype=np.float64)\n        y=np.asarray(y)\n        if y.ndim != 1 or len(y) != len(X) or set(np.unique(y)) != {0,1}:\n            raise ValueError(\'y must contain binary labels 0 and 1, one per row\')\n        if X.ndim!=2 or not np.isfinite(X).all():\n            raise ValueError(\'X must be a finite numeric 2D matrix\')\n        if X_val is None:\n            X,X_val,y,y_val=train_test_split(X,y,test_size=.2,stratify=y,random_state=self.random_state)\n        else:\n            X_val=np.asarray(X_val,dtype=np.float64)\n            y_val=np.asarray(y_val)\n        if (X_val.ndim != 2 or X_val.shape[1] != X.shape[1] or\n                y_val.ndim != 1 or len(y_val) != len(X_val)):\n            raise ValueError(\'Validation rows and feature count must match\')\n        if not np.isfinite(X_val).all() or set(np.unique(y))!={0,1} or set(np.unique(y_val))!={0,1}:\n            raise ValueError(\'both classes required and all features must be finite\')\n        # A threshold calibrated for a previous fit does not apply to the new model.\n        for name in list(vars(self)):\n            if name == \'decision_threshold_\' or name.startswith(\'calibration_\'):\n                delattr(self, name)\n        self.scaler_=StandardScaler().fit(X)\n        Xt=self._features(X); Xv=self._features(X_val)\n        self.linear_=LogisticRegression(C=self.linear_C,solver=\'lbfgs\',max_iter=500,\n                                       random_state=self.random_state).fit(Xt,y)\n        F=self.linear_.decision_function(Xt).astype(np.float64)\n        Fv=self.linear_.decision_function(Xv).astype(np.float64)\n        self.steps_=[]\n        self.pairs_=np.array([(i,j) for i in range(X.shape[1])\n                              for j in range(i+1,X.shape[1])],dtype=np.int64).reshape(-1,2)\n        if self.pairs_.size and self.use_cross:\n            ii,jj=self.pairs_[:,0],self.pairs_[:,1]\n            raw=Xt[:,ii]*Xt[:,jj]\n            self.cross_means_=raw.mean(axis=0)\n            self.cross_stds_=np.maximum(raw.std(axis=0),.05)\n            Ct=self._cross_matrix(Xt);Cv=self._cross_matrix(Xv)\n        else:\n            self.cross_means_=np.array([]);self.cross_stds_=np.array([])\n            Ct=Cv=None\n        best=self._validation_objective(y_val,Fv)\n        best_i=0\n        patience=0\n        self.validation_history_=[]\n        for t in range(self.n_estimators):\n            p=expit(np.clip(F,-25,25))\n            weights=self._weights(y,F,t)\n            # Diagonal Newton approximation for a weighted logistic loss.\n            grad=(y-p)*weights\n            hess=np.maximum(p*(1-p),0.002)*weights\n            response=np.clip(grad/hess,-15,15)\n            tree=DecisionTreeRegressor(max_leaf_nodes=self.max_leaf_nodes,\n                min_samples_leaf=self.min_samples_leaf,\n                random_state=self.random_state+t)\n            tree.fit(Xt,response,sample_weight=hess)\n            F+=self.learning_rate*np.clip(tree.predict(Xt),-5,5)\n            Fv+=self.learning_rate*np.clip(tree.predict(Xv),-5,5)\n            cross_info=None\n            if Ct is not None and Ct.shape[1]:\n                p=expit(np.clip(F,-25,25))\n                weights=self._weights(y,F,t)\n                grad=(y-p)*weights\n                hess=np.maximum(p*(1-p),0.002)*weights\n                num=Ct.T@grad\n                den=(Ct**2).T@hess+self.pair_penalty\n                gain=num*num/den\n                j=int(np.argmax(gain))\n                coefficient=np.clip(num[j]/den[j],-3,3)*self.learning_rate*self.pair_rate\n                F+=coefficient*Ct[:,j]\n                Fv+=coefficient*Cv[:,j]\n                cross_info=(j,float(coefficient))\n            self.steps_.append((tree,cross_info))\n            current=self._validation_objective(y_val,Fv)\n            self.validation_history_.append(current)\n            if current > best+0.00001:\n                best=current;best_i=t+1;patience=0\n            else:\n                patience+=1\n            if patience>=self.early_stopping_rounds:break\n        self.best_iteration_=best_i\n        self.validation_objective_=best\n        self.compile()\n        self.validation_logloss_=self._loss(y_val,self.decision_function(X_val))\n        return self\n', 'discovery.py': '"""Read-only, aggregate discovery for the public ULB Kaggle dataset."""\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport numpy as np\nimport pandas as pd\n\nFEATURES = [f\'V{i}\' for i in range(1, 29)] + [\'Amount\']\n\ndef load_data(path):\n    frame = pd.read_csv(path, usecols=[\'Time\', \'Class\'] + FEATURES)\n    if frame.isna().any().any() or not np.isfinite(frame.to_numpy()).all():\n        raise ValueError(\'Missing or nonfinite values require an explicit preprocessing plan\')\n    if set(frame.Class.unique()) != {0, 1}:\n        raise ValueError(\'Expected both binary labels\')\n    # Identical records are excluded once before partitioning. No rows are exported.\n    duplicate_count = int(frame.duplicated().sum())\n    frame = frame.drop_duplicates().sort_values(\'Time\', kind=\'stable\').reset_index(drop=True)\n    unique_times = np.sort(frame.Time.unique())\n    edges = [unique_times[int(len(unique_times) * q)] for q in (.55, .70, .85)]\n    groups = np.searchsorted(edges, frame.Time.to_numpy(), side=\'right\')\n    parts = [np.flatnonzero(groups == i) for i in range(4)]\n    if any(set(frame.iloc[idx].Class.unique()) != {0, 1} for idx in parts):\n        raise ValueError(\'Every temporal partition needs both classes\')\n    report = dict(source=\'https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud\',\n                  observed_utc=datetime.now(timezone.utc).isoformat(),\n                  sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest(),\n                  scope=\'Complete CSV; identical complete rows removed; time boundaries keep equal timestamps together\',\n                  unit=\'transaction row; identities unavailable\', duplicates_removed=duplicate_count,\n                  rows_after_dedup=len(frame), features=FEATURES,\n                  splits={name: dict(rows=len(idx), positive=int(frame.iloc[idx].Class.sum()),\n                                    negative=int((frame.iloc[idx].Class == 0).sum()),\n                                    time_min=float(frame.iloc[idx].Time.min()),\n                                    time_max=float(frame.iloc[idx].Time.max()))\n                          for name, idx in zip([\'train\', \'validation\', \'calibration\', \'test\'], parts)})\n    return frame, parts, report\n\nif __name__ == \'__main__\':\n    import argparse\n    p = argparse.ArgumentParser(); p.add_argument(\'csv\'); p.add_argument(\'output\')\n    args = p.parse_args()\n    _, _, report = load_data(args.csv)\n    Path(args.output).write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2))\n', 'evaluation.py': '"""Shared calibration and held-out metrics for all benchmark families."""\nimport numpy as np\nfrom scipy.stats import beta\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\ndef calibrate(y, score, alpha=.001, confidence=.95):\n    y, score = np.asarray(y), np.asarray(score, dtype=float)\n    if y.ndim != 1 or score.shape != y.shape or not np.isfinite(score).all():\n        raise ValueError(\'Expected equally sized finite one-dimensional arrays\')\n    if not set(np.unique(y)).issubset({0, 1}) or not 0 < alpha < 1 or not 0 < confidence < 1:\n        raise ValueError(\'Invalid labels or calibration settings\')\n    neg = np.sort(score[y == 0])[::-1]\n    if not len(neg):\n        raise ValueError(\'Calibration requires negative examples\')\n    # This bound assumes independent calibration negatives with the future negative distribution.\n    k = np.arange(int(np.floor(alpha * len(neg))) + 1)\n    upper = beta.ppf(confidence, k + 1, len(neg) - k)\n    feasible = k[upper <= alpha]\n    if not len(feasible):\n        raise ValueError(\'Insufficient calibration negatives for requested confidence\')\n    allowed = int(feasible[-1])\n    return float(np.nextafter(neg[allowed], np.inf)), dict(\n        negative_count=len(neg), allowed=allowed,\n        tolerance_bound=float(upper[allowed]), alpha=alpha, confidence=confidence)\n\ndef metrics(y, score, threshold, alpha=.001):\n    y, score = np.asarray(y), np.asarray(score)\n    flags = score >= threshold\n    pos = y == 1; neg = y == 0\n    tp = int((flags & pos).sum()); fp = int((flags & neg).sum())\n    p, n = int(pos.sum()), int(neg.sum())\n    if not p or not n:\n        raise ValueError(\'Test metrics require both classes\')\n    return dict(ap=float(average_precision_score(y, score)),\n                pauc=float(roc_auc_score(y, score, max_fpr=alpha)),\n                tp=tp, fp=fp, fn=p-tp, tn=n-fp, recall=tp/p, fpr=fp/n,\n                recall_lower_95=float(beta.ppf(.025,tp,p-tp+1)) if tp else 0.,\n                recall_upper_95=float(beta.ppf(.975,tp+1,p-tp)) if tp < p else 1.,\n                fpr_upper_95=float(beta.ppf(.95, fp+1, n-fp)) if fp < n else 1.,\n                precision=tp/max(tp+fp, 1), alerts=tp+fp)\n', 'run_experiment.py': '"""CPU benchmark with disjoint temporal model selection, calibration and test."""\nimport argparse\nimport importlib.metadata\nimport json\nimport platform\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom threadpoolctl import threadpool_limits\nfrom discovery import FEATURES, load_data\nfrom evaluation import calibrate, metrics\n\ndef candidates(seed, trees, full=True):\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    yield \'Logistic\', \'linear\', make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))\n    if not full:\n        return\n    from altcboost_v2 import ALTCBoostV2\n    from altcboost_v3 import ALTCBoostV3\n    from xgboost import XGBClassifier\n    from lightgbm import LGBMClassifier\n    from catboost import CatBoostClassifier\n    # Each tree family gets two prespecified candidates and the same tree limit.\n    for depth in (3, 5):\n        yield \'XGBoost\', f\'depth{depth}\', XGBClassifier(n_estimators=trees, max_depth=depth,\n            learning_rate=.06, n_jobs=2, tree_method=\'hist\', random_state=seed)\n    for leaves in (15, 31):\n        yield \'LightGBM\', f\'leaves{leaves}\', LGBMClassifier(n_estimators=trees,\n            num_leaves=leaves, learning_rate=.06, n_jobs=2, verbosity=-1, random_state=seed)\n    for depth in (3, 5):\n        yield \'CatBoost\', f\'depth{depth}\', CatBoostClassifier(iterations=trees,\n            depth=depth, learning_rate=.06, thread_count=2, verbose=False,\n            allow_writing_files=False, random_seed=seed)\n    common = dict(n_estimators=trees, random_state=seed, min_samples_leaf=40)\n    for cross in (False, True):\n        yield \'ALTCBoostV2\', f\'cross{cross}\', ALTCBoostV2(**common, use_cross=cross)\n        yield \'ALTCBoostV3_original\', f\'cross{cross}\', ALTCBoostV3(**common, use_cross=cross)\n        # A proposed ablation, not an implementation of an exact pAUC objective.\n        yield \'ALTCBoostV3_aligned\', f\'cross{cross}\', ALTCBoostV3(**common,\n            use_cross=cross, hard_negative_strength=0, low_fpr_validation=.001,\n            validation_ap_weight=.5)\n\ndef run(csv, output, seeds=(17, 42), trees=70, full=True):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    frame, parts, discovery = load_data(csv)\n    (output/\'discovery.json\').write_text(json.dumps(discovery, indent=2))\n    arrays = [(frame.iloc[i][FEATURES].to_numpy(dtype=float), frame.iloc[i].Class.to_numpy()) for i in parts]\n    (xt, yt), (xv, yv), (xc, yc), (xe, ye) = arrays\n    rows = []; selections = []\n    for seed in seeds:\n        best = {}\n        for family, variant, model in candidates(seed, trees, full):\n            started = time.perf_counter()\n            with threadpool_limits(limits=2):\n                if family.startswith(\'ALTC\'):\n                    model.fit(xt, yt, xv, yv)\n                else:\n                    model.fit(xt, yt)\n            elapsed = time.perf_counter()-started\n            pv = model.predict_proba(xv)[:, 1]\n            objective = .5*average_precision_score(yv, pv) + .5*roc_auc_score(yv, pv, max_fpr=.001)\n            selections.append(dict(seed=seed, family=family, variant=variant,\n                                   validation_objective=float(objective), fit_s=elapsed))\n            print(seed, family, variant, \'validation\', round(objective, 4), flush=True)\n            if family not in best or objective > best[family][0]:\n                best[family] = (objective, variant, model, elapsed)\n        # Calibrate and touch the test set only after validation selection.\n        for family, (_, variant, model, elapsed) in best.items():\n            pc = model.predict_proba(xc)[:, 1]\n            pe = model.predict_proba(xe)[:, 1]\n            model.predict_proba(xe[:1])\n            single = []\n            for row in xe[:200]:\n                start = time.perf_counter_ns(); model.predict_proba(row.reshape(1, -1))\n                single.append((time.perf_counter_ns()-start)/1000)\n            batch = []\n            for _ in range(5):\n                start = time.perf_counter(); model.predict_proba(xe)\n                batch.append((time.perf_counter()-start)*1e6/len(xe))\n            for alpha in (.0001, .001, .01):\n                try:\n                    cut, details = calibrate(yc, pc, alpha)\n                except ValueError as e:\n                    rows.append(dict(seed=seed, method=family, variant=variant, alpha=alpha,\n                                     status=\'unsupported\', reason=str(e)))\n                    continue\n                row = dict(seed=seed, method=family, variant=variant,\n                    status=\'evaluated\', fit_s=elapsed, threshold=cut,\n                    batch_us_per_row=float(np.median(batch)), single_p50_us=float(np.median(single)),\n                    single_p99_us=float(np.quantile(single, .99)), **details,\n                    **metrics(ye, pe, cut, alpha))\n                rows.append(row)\n                print(family, alpha, \'recall\', round(row[\'recall\'], 4), \'FPR\', round(row[\'fpr\'], 6), flush=True)\n        pd.DataFrame(rows).to_csv(output/\'results.csv\', index=False)\n        pd.DataFrame(selections).to_csv(output/\'validation.csv\', index=False)\n    packages=[\'numpy\',\'scipy\',\'scikit-learn\',\'pandas\']\n    if full:\n        packages+=[\'numba\',\'xgboost\',\'lightgbm\',\'catboost\']\n    versions = {p: importlib.metadata.version(p) for p in packages}\n    (output/\'environment.json\').write_text(json.dumps(dict(python=platform.python_version(),\n        platform=platform.platform(), versions=versions, seeds=list(seeds), trees=trees,\n        note=\'Temporal split is fixed across seeds; seeds are model randomness, not independent datasets\'), indent=2))\n    return rows\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(); p.add_argument(\'csv\'); p.add_argument(\'--output\', default=\'results\')\n    p.add_argument(\'--trees\', type=int, default=70); p.add_argument(\'--seeds\', nargs=\'+\', type=int, default=[17,42])\n    args = p.parse_args(); run(args.csv, args.output, tuple(args.seeds), args.trees)\n', 'tests/test_unit.py': "import numpy as np\nimport pytest\nfrom scipy.special import expit\nfrom sklearn.datasets import make_classification\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.tree import DecisionTreeRegressor\nfrom altcboost import ALTCBoostClassifier\nfrom altcboost_v2 import ALTCBoostV2\nfrom altcboost_v3 import ALTCBoostV3\nfrom evaluation import calibrate, metrics\n\ndef test_calibration_ties_and_confidence():\n    score = np.ones(40000) * .2\n    cut, info = calibrate(np.zeros(len(score)), score, .0001)\n    assert not (score >= cut).any()\n    assert info['tolerance_bound'] <= .0001\n\ndef test_calibration_rejects_small_holdout():\n    with pytest.raises(ValueError, match='Insufficient'):\n        calibrate(np.zeros(100), np.linspace(0,1,100), .0001)\n\ndef test_metrics_count_actual_errors():\n    result = metrics([0,0,1,1], [.1,.8,.6,.9], .7)\n    assert (result['fp'], result['tp'], result['fn'], result['tn']) == (1,1,1,1)\n\ndef fitted(model_type=ALTCBoostV3):\n    x,y = make_classification(n_samples=400,n_features=4,n_informative=3,n_redundant=0,random_state=4)\n    a,b,c,d = train_test_split(x,y,test_size=.3,random_state=1,stratify=y)\n    model = model_type(n_estimators=5,min_samples_leaf=10).fit(a,c,b,d)\n    return model,a,c,b,d\n\ndef test_validation_loss_matches_selected_model():\n    m,_,_,x,y = fitted()\n    assert m.validation_logloss_ == pytest.approx(m._loss(y,m.decision_function(x)))\n\n@pytest.mark.parametrize('model_type',[ALTCBoostV2,ALTCBoostV3])\ndef test_refit_invalidates_threshold(model_type):\n    m,x,y,xv,yv = fitted(model_type)\n    m.fit_threshold(xv,yv)\n    m.fit(x,y,xv,yv)\n    with pytest.raises(ValueError,match='fit_threshold'):\n        m.predict_flags(xv)\n\n@pytest.mark.parametrize('bad', [np.full(400,.9), np.array([0.,1.9]*200)])\ndef test_fractional_labels_are_rejected_before_cast(bad):\n    with pytest.raises(ValueError,match='binary labels'):\n        ALTCBoostV3().fit(np.ones((400,4)),bad)\n\ndef test_prediction_rejects_feature_mismatch():\n    m,*_ = fitted()\n    with pytest.raises(ValueError,match='feature count'):\n        m.predict_proba(np.ones((2,3)))\n\ndef test_compiled_tree_boundary_matches_sklearn():\n    m,_,_,_,_ = fitted()\n    # Construct a split where float32 rounding changes the branch.\n    tree = DecisionTreeRegressor(max_depth=1).fit([[0,0,0,0],[2,0,0,0]],[-2,2])\n    m.steps_=[(tree,None)]; m.best_iteration_=1; m.compile()\n    x = m._means.copy().reshape(1,-1)\n    x[0,0] += m._scales[0] * (1 + 1e-8)\n    expected = ALTCBoostClassifier.decision_function(m,x)\n    assert np.allclose(m.decision_function(x),expected,rtol=0,atol=1e-12)\n    m.decision_threshold_=float(expit(expected[0]-.01))\n    flags,_=m.predict_flags(x)\n    assert flags[0] == 1\n", 'tests/test_integration.py': "import json\nimport numpy as np\nimport pandas as pd\nfrom discovery import FEATURES, load_data\nfrom run_experiment import run\n\ndef fixture_csv(tmp_path):\n    rng=np.random.default_rng(42); n=24000\n    data=pd.DataFrame(rng.normal(size=(n,len(FEATURES))),columns=FEATURES)\n    data['Time']=np.arange(n)//2\n    data['Class']=(rng.random(n)<.04).astype(int)\n    path=tmp_path/'synthetic.csv'; data.to_csv(path,index=False)\n    return path\n\ndef test_temporal_partitions_keep_equal_timestamps_together(tmp_path):\n    frame,parts,_=load_data(fixture_csv(tmp_path))\n    assert len(set(np.concatenate(parts))) == len(frame)\n    for a,b in zip(parts,parts[1:]):\n        assert frame.iloc[a].Time.max() < frame.iloc[b].Time.min()\n\ndef test_user_receives_heldout_results_and_unsupported_budget(tmp_path):\n    output=tmp_path/'out'\n    rows=run(fixture_csv(tmp_path),output,seeds=(17,),trees=2,full=False)\n    assert len(rows)==3\n    assert next(r for r in rows if r['alpha']==.0001)['status']=='unsupported'\n    assert next(r for r in rows if r['alpha']==.01)['status']=='evaluated'\n    assert (output/'results.csv').exists()\n    assert json.loads((output/'discovery.json').read_text())['unit'].startswith('transaction row')\n\ndef test_model_training_calibration_and_flags_on_disjoint_rows():\n    from sklearn.datasets import make_classification\n    from altcboost_v3 import ALTCBoostV3\n    x,y=make_classification(n_samples=1600,n_features=6,random_state=12)\n    model=ALTCBoostV3(n_estimators=8,min_samples_leaf=15)\n    model.fit(x[:700],y[:700],x[700:1000],y[700:1000])\n    model.fit_threshold(x[1000:1300],y[1000:1300],max_fpr=.05)\n    flags,trees=model.predict_flags(x[1300:])\n    assert np.array_equal(flags,model.predict_proba(x[1300:])[:,1]>=model.decision_threshold_)\n    assert (trees<=model.best_iteration_).all()\n    model.fit(x[:700],y[:700],x[700:1000],y[700:1000])\n    model.fit_threshold(x[1000:1300],y[1000:1300],max_fpr=.05)\n    assert model.predict_flags(x[1300:])[0].shape==(300,)\n"}
for name,content in files.items():
    path=root/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content)
sys.path.insert(0,str(root))
os.chdir(root)


In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_unit.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_integration.py'],check=True)


In [ ]:
from discovery import load_data
matches=list(Path('/kaggle/input').rglob('creditcard.csv'))
if len(matches)!=1:
    raise RuntimeError('Attach exactly one mlg-ulb/creditcardfraud dataset containing creditcard.csv')
csv_path=matches[0]
_,_,discovery=load_data(csv_path)
print(json.dumps(discovery,indent=2))


## Prespecified experiment

The complete dataset is split into four time intervals. Equal timestamps stay together.
Identical complete rows are removed as a benchmark preprocessing choice, not evidence that those transactions were invalid.
The models see only the training interval. Validation selects configurations; a later calibration interval sets thresholds.
The final interval is used for reported results. Every model uses the same calibration rule.

The default below uses two seeds and a 70-tree limit. These are two training runs on one fixed split.
They do not constitute independent datasets. Equal configuration counts and tree limits do not establish equal tuning effort.
ALTCBoost uses internal early stopping; library baselines use their complete tree budget.


In [ ]:
from run_experiment import run
rows=run(csv_path, '/kaggle/working/altcboost_results', seeds=(17,42), trees=70)


In [ ]:
import pandas as pd
result=pd.read_csv('/kaggle/working/altcboost_results/results.csv')
display(result[['method','seed','alpha','status','recall','fpr','fpr_upper_95','precision','fit_s','single_p50_us','single_p99_us']])
print('Calibration bounds assume representative independent negatives. Temporal drift can violate that assumption.')
print('Latency measures model scoring on this machine; feature retrieval and network latency are excluded.')


## Research behind the experiment

* [Partial AUC minimax formulations](https://arxiv.org/abs/2512.01213): principled training objectives differ from the heuristic implemented here.
* [Large-scale partial AUC optimization](https://arxiv.org/abs/2203.01505): focuses ranking on a specified false-positive interval.
* [Neyman-Pearson classification](https://arxiv.org/abs/1802.02558): error control requires explicit distributional assumptions.
* [TabArena](https://arxiv.org/abs/2506.16791): validation, tuning and ensembles affect model comparisons.

Choose a candidate on validation data before inspecting final results. Repeatedly redesigning against the final holdout turns it into another validation set.
